In [ ]:


## Packages ---

import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
import time
import functools as ft
from IPython.display import display


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'Census'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_dr = path_prod / 'Small Data Requests' / date.today().strftime('%Y')
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_out_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")




In [ ]:




    
def clean_pop_6(df_census, params):

    ## TODO: move to a different python file

    # Pop_6 is unique, needs it's own function
    file_weights = PATH_WEIGHTS / f'Pop_3 {params['geo']} {params['estimate']}.xlsx'
    df_pop = pd.read_excel(file_weights, sheet_name=GEO_SHEETS[params['geo']])

    df_pop = df_pop.rename(columns={'Population': f'Total {params['metric']}'})

    conditions = [
                    (df_pop["Race/Ethnicity"] == 'All'                                            ),
                    (df_pop["Race/Ethnicity"] == 'American Indian or Alaska Native (NH)'          ),
                    (df_pop["Race/Ethnicity"] == 'Asian (NH)'                                     ),
                    (df_pop["Race/Ethnicity"] == 'Black or African American (NH)'                 ),
                    (df_pop["Race/Ethnicity"] == 'Hispanic or Latino'                             ),
                    (df_pop["Race/Ethnicity"] == 'Native Hawaiian or other Pacific Islander (NH)' ),
                    (df_pop["Race/Ethnicity"] == 'White (NH)'                                     ),
                    (df_pop["Race/Ethnicity"] == 'Some other race (NH)'                           ),
                    (df_pop["Race/Ethnicity"] == 'Two or more races (NH)'                         )
                ]
    choices = ["All", "American Indian or Alaska Native", "Asian", "Black or African American", "Hispanic or Latino",
                "Native Hawaiian or other Pacific Islander", "White (NH)", "Some other race", "Two or more races"]
    df_pop["Race/Ethnicity"] = np.select(conditions, choices)

    if params['geo'] == 'MSA': field_id = 'MSA_ID'
    if params['geo'] == 'MPO': field_id = 'MPO'
    else: field_id ='NAME'

    df_census = df_census.merge(df_pop, on=[field_id, 'Year', 'Race/Ethnicity'], how='left')

    df_census['Total Population'] = df_census['Total Population'].fillna(0)
    df_census['Total Population'] = df_census['Total Population'].replace(0, 1)
    df_census['Birth Rate Per 1,000 People'] = (df_census['Population'] / df_census['Total Population']) * 1000
    df_census = df_census[df_census['Variable'] == 'Woman aged 15-44 who had a birth in the past 12 months']
    
    return df_census


def clean_pop_7(df_census, params):

    ## TODO: move to a different python file

    # Pop_7 is unique, needs it's own function
    geo_ID = GEOID_CLEAN[params['geo']]
    df_census['Total Population'] = df_census.groupby(geo_ID + ['Year'])['Population'].transform('sum')

    df_census['Marriage Rate Per 1,000 People'] = (df_census['Population'] / df_census['Total Population']) * 1000
    df_census = df_census[df_census['Variable'] == 'Married last year']
    df_census = df_census.drop_duplicates().reset_index(drop=True)
    
    return df_census



Census Tracts

In [ ]:


## Import ---

path_in  = path_out_server
workbook = 'Pop_6 Tracts ACS5.xlsx'
sheet_name = 'Tracts'

file_pop6 = path_in / workbook
df_pop6 = pd.read_excel(file_pop6, sheet_name=sheet_name)

# Clean
if 'National' not in df_pop6.columns:
    path_nat = path_main / 'Vibrant and Inclusive Places' / 'People and Community' / 'Pop and Demographics' / 'Pop_6 Birth Rates'
    file_nat = path_nat / 'Pop_6 National ACS5.xlsx'
    df_nat = pd.read_excel(file_nat, sheet_name = 'National')
    df_nat = df_nat[['Year', 'Birth Rate Per 1,000 People']].rename(columns={'Birth Rate Per 1,000 People':'National'})
    df_pop6 = df_pop6.merge(df_nat, on='Year', how='left')
    df_pop6 = df_pop6.drop_duplicates()

    conditions = [
          df_pop6['Birth Rate Per 1,000 People'] == df_pop6['National']
        , df_pop6['Birth Rate Per 1,000 People']  > df_pop6['National']
        , df_pop6['Birth Rate Per 1,000 People']  < df_pop6['National']
        , df_pop6['Birth Rate Per 1,000 People'].isna()
    ]    
    choices = ['Equal to national birth rate', 'Higher than national birth rate', 'Lower than national birth rate', 'No Data']
    df_pop6['National Comparison'] = np.select(conditions, choices, default='missing conditions')
    
    df_pop6 = df_pop6[df_pop6['Year'] == 2023]

    df_pop6 = df_pop6.reset_index(drop=True)
    
display(df_pop6['National Comparison'].value_counts())
display(df_pop6.head())


## Export ---

# Set inputs and outputs
export=False
path_out = path_dr / 'Opportunity Zones'


# Export function
def export_to_csv(df):
    global workbook
    
    df.columns = [x.lower() for x in df.columns]
    df.columns = [re.sub('[^\\w\\s]', '_', col.strip()) for col in df.columns]
    df.columns = [re.sub('[\s+]'    , '_', col.strip()) for col in df.columns]
    df.columns = [re.sub('\\?'      , '' , col.strip()) for col in df.columns]
    workbook = re.sub('[\s+]', '_'   , workbook)
    workbook = re.sub('.xlsx', '.csv', workbook)
    workbook = workbook.lower()

    file_out = path_out / workbook
    print('Exporting here: ', path_out)
    print('Name of export: ', workbook)
    display(df.head())
    if export:
        df.to_csv(file_out, index=False)


export_to_csv(df_pop6)



In [ ]:


## Import ---

path_in  = path_out_server
workbook = 'Pop_8 Tracts ACS5.xlsx'
sheet_name = 'Tracts'

file_pop8 = path_in / workbook
df_pop8 = pd.read_excel(file_pop8, sheet_name=sheet_name)

# Clean
if 'National' not in df_pop6.columns:
    path_nat = path_main / 'Vibrant and Inclusive Places' / 'People and Community' / 'Pop and Demographics' / 'Pop_8 Marriage Status'
    file_nat = path_nat / 'Pop_8 National ACS5.xlsx'
    df_nat = pd.read_excel(file_nat, sheet_name = 'National')
    df_nat = df_nat[['Year', 'Variable', 'Percentage']].rename(columns = {'Percentage':'National'})
    df_pop8 = df_pop8.merge(df_nat, on=['Year', 'Variable'], how='left')
    df_pop8 = df_pop8.drop_duplicates()

    conditions = [
          df_pop8['Percentage'] == df_pop8['National']
        , df_pop8['Percentage']  > df_pop8['National']
        , df_pop8['Percentage']  < df_pop8['National']
        , df_pop8['Percentage'].isna()
    ]    
    choices = ['Equal to national marriage proportion', 'Higher than national marriage proportion', 'Lower than national marriage proportion', 'No Data']
    df_pop8['National Comparison'] = np.select(conditions, choices, default='missing conditions')
    
    df_pop8 = df_pop8[df_pop8['Year'] == 2023]
    df_pop8 = df_pop8[df_pop8['Variable'] == 'Now married']

    df_pop8 = df_pop8.reset_index(drop=True)
    
display(df_pop8['National Comparison'].value_counts())
display(df_pop8.head())


## Export ---

# Set inputs and outputs
export=False
path_out = path_dr / 'Opportunity Zones'


# Export function
def export_to_csv(df):
    global workbook
    
    df.columns = [x.lower() for x in df.columns]
    df.columns = [re.sub('[^\\w\\s]', '_', col.strip()) for col in df.columns]
    df.columns = [re.sub('[\s+]'    , '_', col.strip()) for col in df.columns]
    df.columns = [re.sub('\\?'      , '' , col.strip()) for col in df.columns]
    workbook = re.sub('[\s+]', '_'   , workbook)
    workbook = re.sub('.xlsx', '.csv', workbook)
    workbook = workbook.lower()

    file_out = path_out / workbook
    print('Exporting here: ', path_out)
    print('Name of export: ', workbook)
    display(df.head())
    if export:
        df.to_csv(file_out, index=False)


export_to_csv(df_pop8)



ZIP Codes

In [ ]:


## Import ---

path_in  = path_out_server
workbook = '20241030_births_final_zip_year_sup.csv'

file_pop6 = path_in / workbook
df_pop6 = pd.read_csv(file_pop6)


# Clean
df_pop6 = df_pop6[df_pop6['Year'] == 2023]
df_pop6 = df_pop6[df_pop6['Geography'] == 'Residence']


path_nat = path_main / 'Vibrant and Inclusive Places' / 'People and Community' / 'Pop and Demographics' / 'Pop_6 Birth Rates'
file_nat = path_nat / 'Pop_6 National ACS5.xlsx'
df_nat = pd.read_excel(file_nat, sheet_name = 'National')
df_nat = df_nat[['Year', 'Birth Rate Per 1,000 People']].rename(columns={'Birth Rate Per 1,000 People':'National'})
df_pop6 = df_pop6.merge(df_nat, on='Year', how='left')
df_pop6 = df_pop6.drop_duplicates()

conditions = [
      df_pop6['Birth Rate Per 1,000 People'] == df_pop6['National']
    , df_pop6['Birth Rate Per 1,000 People']  > df_pop6['National']
    , df_pop6['Birth Rate Per 1,000 People']  < df_pop6['National']
    , df_pop6['Birth Rate Per 1,000 People'].isna()
]    
choices = ['Equal to national birth rate', 'Higher than national birth rate', 'Lower than national birth rate', 'No Data']
df_pop6['National Comparison'] = np.select(conditions, choices, default='missing conditions')

df_pop6 = df_pop6[df_pop6['Year'] == 2023]

df_pop6 = df_pop6.reset_index(drop=True)
    
display(df_pop6['National Comparison'].value_counts())
display(df_pop6.head())


## Export ---

# Set inputs and outputs
export=False
path_out = path_dr / 'Opportunity Zones'


# Export function
def export_to_csv(df):
    global workbook
    
    df.columns = [x.lower() for x in df.columns]
    df.columns = [re.sub('[^\\w\\s]', '_', col.strip()) for col in df.columns]
    df.columns = [re.sub('[\s+]'    , '_', col.strip()) for col in df.columns]
    df.columns = [re.sub('\\?'      , '' , col.strip()) for col in df.columns]
    workbook = re.sub('[\s+]', '_'   , workbook)
    workbook = re.sub('.xlsx', '.csv', workbook)
    workbook = workbook.lower()

    file_out = path_out / workbook
    print('Exporting here: ', path_out)
    print('Name of export: ', workbook)
    display(df.head())
    if export:
        df.to_csv(file_out, index=False)


export_to_csv(df_pop6)

